# 🏦 リース審査 TimesFM バックエンド (Google Colab)

このノートブックは **ローカルの Streamlit アプリ** から呼び出される FastAPI サーバーを
Colab 上で起動し、ngrok で公開します。

### 自動連携の仕組み
1. Colab が ngrok URL を **Google Drive** のファイルに自動保存
2. ローカルの Mac（Google Drive for Desktop が必要）がそのファイルを検出
3. Streamlit アプリが URL を自動取得 → 手動コピペ不要

### 使い方
1. ランタイム → **「T4 GPU」** を選択
2. **「すべてのセルを実行」** するだけ
3. ローカルの Streamlit「📊 3期財務分析」で **「🔄 Colab URL を自動取得」** をクリック

In [ ]:
# ── セル 1: 依存パッケージのインストール ─────────────────────────────────
!pip install -q timesfm fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# ── セル 2: Google Drive のマウント ──────────────────────────────────────
# URL の自動同期に使用。「Googleドライブに接続」ボタンが表示されたら許可してください。
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive マウント完了')

In [ ]:
# ── セル 3: ngrok トークンの設定 ──────────────────────────────────────────
# https://dashboard.ngrok.com/get-started/your-authtoken から取得（無料アカウントで OK）
NGROK_TOKEN = ""  # ← ngrok トークンをここに貼り付ける

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✅ ngrok トークン設定済み')
else:
    print('⚠️ NGROK_TOKEN 未設定（トークンなしでも動作しますが不安定になる場合があります）')

In [ ]:
# ── セル 4: FastAPI サーバーコード ────────────────────────────────────────
BACKEND_CODE = '''
from __future__ import annotations
import math
from datetime import date
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, field_validator

try:
    import timesfm, pandas as pd
    _TFM = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(backend="gpu", per_core_batch_size=32, horizon_len=128),
        checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id="google/timesfm-1.0-200m"),
    )
    TIMESFM_AVAILABLE = True
    print("✅ TimesFM 初期化完了")
except Exception as e:
    _TFM = None; TIMESFM_AVAILABLE = False
    print(f"⚠️ TimesFM フォールバック: {e}")

SEASONAL_INDICES = {
    "建設業":       [0.6, 0.7, 1.8, 0.8, 0.9, 0.9, 0.8, 0.9, 1.0, 0.9, 1.0, 1.7],
    "小売業":       [0.9, 0.8, 1.0, 0.9, 0.9, 0.9, 1.0, 1.0, 0.9, 1.0, 1.1, 1.6],
    "製造業":       [0.9, 0.9, 1.1, 1.0, 1.0, 1.0, 1.0, 0.9, 1.1, 1.0, 1.0, 1.1],
    "卸売業":       [0.9, 0.9, 1.2, 1.0, 1.0, 1.0, 1.0, 0.9, 1.0, 1.0, 1.0, 1.1],
    "医療・福祉":   [1.0]*12,
    "飲食・宿泊業": [0.8, 0.8, 1.0, 1.0, 1.1, 1.1, 1.3, 1.2, 1.0, 1.0, 0.9, 0.8],
    "サービス業":   [0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.1],
    "不動産業":     [0.8, 0.9, 1.4, 1.1, 1.0, 0.9, 0.9, 0.9, 1.0, 0.9, 0.9, 1.3],
    "情報通信業":   [1.0]*12,
    "運輸・物流":   [0.9, 0.9, 1.1, 1.0, 1.0, 0.9, 0.9, 1.0, 1.0, 1.0, 1.1, 1.2],
}

app = FastAPI(title="リース審査 TimesFM API (Colab)")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ForecastRequest(BaseModel):
    sales: list[float]; profit: list[float]; net_assets: list[float]; industry: str = "サービス業"
    @field_validator("sales","profit","net_assets")
    @classmethod
    def must_be_three(cls, v):
        if len(v) != 3: raise ValueError("3要素必要")
        return v

class ForecastResponse(BaseModel):
    months_history: list[str]; sales_history: list[float]; profit_history: list[float]
    net_assets_history: list[float]; months_forecast: list[str]; sales_forecast: list[float]
    profit_forecast: list[float]; net_assets_forecast: list[float]; timesfm_available: bool

def _annual_to_monthly(annual, industry):
    idx = SEASONAL_INDICES.get(industry, [1.0]*12)
    return [annual[i//12]/12.0*idx[i%12] for i in range(36)]

def _timesfm_forecast(history, horizon=12):
    if not TIMESFM_AVAILABLE or _TFM is None: return _gbm(history, horizon)
    try:
        df = pd.DataFrame({"unique_id":["x"]*len(history),"ds":range(len(history)),"y":history})
        fdf, _ = _TFM.forecast_on_df(inputs=df, freq="M", value_name="y", num_jobs=1)
        col = [c for c in fdf.columns if "timesfm" in c.lower()]
        if col: return fdf[col[0]].values[:horizon].tolist()
    except: pass
    return _gbm(history, horizon)

def _gbm(history, horizon=12):
    if not history or history[-1]<=0: return [0.0]*horizon
    arr=np.clip(history,1e-6,None); mu=float(np.mean(np.diff(np.log(arr)))) if len(arr)>1 else 0.01
    return [history[-1]*math.exp(mu*(t+1)/12) for t in range(horizon)]

def _labels(y,m,n):
    out=[]
    for _ in range(n):
        out.append(f"{y:04d}-{m:02d}"); m+=1
        if m>12: m,y=1,y+1
    return out

@app.get("/health")
def health(): return {"status":"ok","timesfm":TIMESFM_AVAILABLE,"backend":"colab"}

@app.post("/forecast", response_model=ForecastResponse)
def forecast(req: ForecastRequest):
    t=date.today(); hl=_labels(t.year-3,t.month,36); fl=_labels(t.year,t.month,12)
    sh=_annual_to_monthly(req.sales,req.industry)
    ph=_annual_to_monthly(req.profit,req.industry)
    nh=_annual_to_monthly(req.net_assets,req.industry)
    return ForecastResponse(
        months_history=hl,sales_history=sh,profit_history=ph,net_assets_history=nh,
        months_forecast=fl,sales_forecast=_timesfm_forecast(sh),
        profit_forecast=_timesfm_forecast(ph),net_assets_forecast=_timesfm_forecast(nh),
        timesfm_available=TIMESFM_AVAILABLE)
'''
with open('/content/backend_colab.py', 'w') as f:
    f.write(BACKEND_CODE)
print('backend_colab.py を作成しました')

In [ ]:
# ── セル 5: サーバー起動 + ngrok 公開 + Drive に URL 自動保存 ─────────────
import nest_asyncio, uvicorn, threading, time, json, importlib.util
from pyngrok import ngrok
from pathlib import Path

nest_asyncio.apply()

# backend_colab.py を動的ロード
spec = importlib.util.spec_from_file_location('backend_colab', '/content/backend_colab.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# uvicorn をバックグラウンドで起動
PORT = 8000
threading.Thread(
    target=lambda: uvicorn.run(mod.app, host='0.0.0.0', port=PORT, log_level='warning'),
    daemon=True
).start()
time.sleep(2)

# ngrok トンネル開通
tunnel     = ngrok.connect(PORT, bind_tls=True)
public_url = tunnel.public_url

# ── Google Drive に URL を自動保存 ──────────────────────────────────────
# Streamlit アプリが「🔄 Colab URL を自動取得」ボタンでこのファイルを読む
DRIVE_FILE = Path('/content/drive/MyDrive/colab_backend_url.json')
url_data = {
    'url':            public_url,
    'timesfm':        mod.TIMESFM_AVAILABLE,
    'saved_at':       time.strftime('%Y-%m-%d %H:%M:%S'),
}
try:
    DRIVE_FILE.write_text(json.dumps(url_data, ensure_ascii=False, indent=2))
    print(f'✅ Google Drive に URL を保存しました → {DRIVE_FILE}')
except Exception as e:
    print(f'⚠️ Drive への保存失敗（手動でURLをコピーしてください）: {e}')

print('=' * 60)
print('✅ TimesFM バックエンド起動完了！')
print(f'   URL: {public_url}')
print()
print('📱 Streamlit での次の操作:')
print('   「📊 3期財務分析」→「⚙️ バックエンド設定」')
print('   →「🔄 Colab URL を自動取得」ボタンをクリック')
print('=' * 60)

In [ ]:
# ── セル 6: 接続テスト（オプション） ─────────────────────────────────────
import requests
r = requests.get(f'{public_url}/health')
print('/health:', r.json())
r2 = requests.post(f'{public_url}/forecast', json={
    'sales':[500000,520000,550000],'profit':[30000,35000,38000],
    'net_assets':[120000,145000,170000],'industry':'建設業'
})
d = r2.json()
print(f'/forecast: timesfm={d["timesfm_available"]}, 12ヶ月後売上={d["sales_forecast"][-1]:,.0f}千円')

In [ ]:
# ── セル 7: サーバーを止めるとき ─────────────────────────────────────────
# ngrok.kill()
# print('サーバーを停止しました')